In [ ]:
import meshio as mio
import h5py
import numpy as np
import pyvista as pv
from ipywidgets import interact, interactive, fixed, interact_manual
from matplotlib.widgets import Button, Slider
import ipywidgets as widgets
import scipy as sp
import matplotlib.pyplot as mplt
import matplotlib
import matplotlib.animation as animation
import json as js
import meshio as mio
import subprocess as sup
import math

pv.set_jupyter_backend('trame')
# pv.set_jupyter_backend('client')



In [ ]:
def to_pyvista_mesh(V, F = None):
    if F is None:
        return pv.PolyData(V)
    if F.shape[1] == 3:
        return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)
    elif F.shape[1] == 4:
        return pv.UnstructuredGrid({pv.CellType.TETRA: F}, V)



In [ ]:
# path = "/Users/teseo/Downloads/Embryogram test/new/0in-analysis-07_09T15_10-analysis.hdf5"
# path = "/Users/teseo/Downloads/Embryogram test/20250510_tracking-analysis-07_19T18_47-analysis.hdf5"
# path = "/Users/zoeli/Documents/UVic/masters/other/fish/still.hdf5"
#path = "/Users/zoeli/Documents/UVic/masters/other/fish/neweststill.hdf5"
path = "/Users/zoeli/Documents/UVic/masters/other/fish/not_control.hdf5"

#polyfem = "/Users/teseo/Documents/scuola/polyfem/polyfem.nosync/bin_rel.nosync/PolyFEM_bin"

In [ ]:
hdf5_file = h5py.File(path, "r")
V, T = hdf5_file["mesh/v"][:].astype(float), hdf5_file["mesh/t"][:].astype(np.int32)

top = hdf5_file["bc/top"][:].astype(np.int32)
bottom = hdf5_file["bc/bottom"][:].astype(np.int32)
middle = hdf5_file["bc/middle"][:].astype(np.int32)

In [ ]:
centers = hdf5_file["bc_func/centers"][:].astype(float)
eps = hdf5_file["bc_func/eps"][()]

In [ ]:
nk = len(hdf5_file["bc_func"].keys())-2

disps = []

for i in range(nk):
    disps.append(hdf5_file[f"bc_func/disp{i+1}"][:].astype(float))

In [ ]:
data = [np.mean(t[0]) for t in disps]
mplt.plot(data, label='x')
data = [np.mean(t[1]) for t in disps]
mplt.plot(data, label='y')
data = [np.mean(t[2]) for t in disps]
mplt.plot(data, label='z')
mplt.legend()
mplt.xlabel('time')
mplt.ylabel('disp')
mplt.title('Disp after before correction')
mplt.show()

In [ ]:
E = hdf5_file["problem/E"][()] .astype(float)
nu = hdf5_file["problem/nu"][()] .astype(float)
is_linear = hdf5_file["problem/is_linear"][()] .astype(bool)

# E

In [ ]:
# m=to_pyvista_mesh(V, T)
# # m=m.explode(0.5)
# plt = pv.Plotter()
# plt.add_mesh(m, show_edges=True, style='wireframe', line_width=1.0)
# plt.add_mesh(to_pyvista_mesh(V[top]), color='red', point_size=10, render_points_as_spheres=True, name='top')
# plt.add_mesh(to_pyvista_mesh(V[bottom]), color='green', point_size=10, render_points_as_spheres=True, name='bottom')
# plt.add_mesh(to_pyvista_mesh(V[middle]), color='blue', point_size=10, render_points_as_spheres=True, name='middle')
# plt.show()

In [ ]:
# m=m.explode(0.5)
plt = pv.Plotter()
plt.add_mesh(to_pyvista_mesh(V[middle]), color='blue', point_size=8, render_points_as_spheres=True, name='middle')
plt.add_mesh(to_pyvista_mesh(centers), color='red', point_size=7, render_points_as_spheres=True, name='middle')
plt.show()

In [ ]:
angles = []
lengths = []

for i in range(len(disps) - 1):
    disps_0 = disps[i]
    disps_1 = disps[i + 1]

    angles_per_frame = []
    lengths_per_frame = []
    for j in range(len(disps_0)):
        v0 = disps_0[j]
        v1 = disps_1[j]
        angle = np.dot(v0, v1) / (np.linalg.norm(v0) * np.linalg.norm(v1))
        angles_per_frame.append(angle)
        lengths_per_frame.append(abs(np.linalg.norm(v0) - np.linalg.norm(v1)) / np.linalg.norm(v0))
    
    angles.append(angles_per_frame)
    lengths.append(lengths_per_frame)

mplt.hist(lengths[100], bins=50, color='blue')
mplt.show()

bad = [np.zeros(len(disps[0]))]

for i, angles_per_frame in enumerate(angles):
    bad_per_frame = bad[i].copy()

    for j, angle in enumerate(angles_per_frame):
        if lengths_per_frame[j] > bad_per_frame[j]:
            bad_per_frame[j] = lengths_per_frame[j]

    bad.append(bad_per_frame)

mplt.hist(bad[100], bins=50, color='blue')
mplt.show()

In [ ]:
lengths = []

for i in range(len(disps) - 1):
    disps_0 = disps[i]
    disps_1 = disps[i + 1]
    lengths_per_frame = []

    for j in range(len(disps_0)):
        v0 = disps_0[j]
        v1 = disps_1[j]
        lengths_per_frame.append(abs(np.linalg.norm(v1) - np.linalg.norm(v0)))
    
    lengths.append(lengths_per_frame)

fig, ax = mplt.subplots()
#ax.autoscale()
ax.set_xlim(0.0, 2.0)
ax.set_ylim(0.0, 700.0)
*_, patches = ax.hist(lengths[0], bins=50, color='blue')
text = ax.text(.01, .99, str(0), ha='left', va='top', transform=ax.transAxes)

def update_histogram(frame):
    global patches
    global text
    for p in patches:
        p.remove()
    text.remove()

    *_, patches = ax.hist(lengths[frame], bins=50, color='blue')
    text = ax.text(.01, .99, str(frame), ha='left', va='top', transform=ax.transAxes)
    return (patches, text)

ani = animation.FuncAnimation(fig=fig, func=update_histogram, frames=len(lengths), interval=90)
ani.save("abs_change_disp_length_over_time.gif")

In [ ]:
neighbourhood_similarity = []
neighbourhood_length_difference = []
neighbourhood_difference = []
neighbourhood_means = []
#neighbourhood_stdevs = []

for i, disps_per_frame in enumerate(disps):
    similarity_per_frame = []
    length_difference_per_frame = []
    difference_per_frame = []
    mean_per_frame = []
    #stdev_per_frame = []

    for j, disp in enumerate(disps_per_frame):
        distances = np.linalg.norm(centers - centers[j], axis=1)
        indices = np.argsort(distances)
        neighbourhood = indices[1:19]
        neighbourhood_disps = disps_per_frame[neighbourhood]

        neighbourhood_mean = np.mean(neighbourhood_disps, axis=0)
        #neighbourhood_std_dev = np.std(neighbourhood_disps, axis=0)
        #z_scores = np.abs((neighbourhood_disps - neighbourhood_mean) / neighbourhood_std_dev, axis=0)

        #print(neighbourhood_disps[z_scores > 3.0])
        #continue

        neighbourhood_lengths = np.linalg.norm(neighbourhood_disps, axis=1)
        neighbourhood_length_mean = np.mean(neighbourhood_lengths)
        cosine_similarity = np.dot(neighbourhood_mean, disp) / (np.linalg.norm(neighbourhood_mean) * np.linalg.norm(disp))
        length_difference = abs(neighbourhood_length_mean - np.linalg.norm(disp))
        #length_difference = abs(np.linalg.norm(neighbourhood_mean) - np.linalg.norm(disp))
        difference = np.linalg.norm(neighbourhood_mean - disp)

        #temp = [np.linalg.norm(neighbourhood_mean) - np.linalg.norm(d) for d in disps_per_frame[neighbourhood]]
        #temp2 = [(diff * diff) / (len(disps_per_frame[neighbourhood]) - 1.0) for diff in temp]
        #std_dev = math.sqrt(sum(temp2))
        
        similarity_per_frame.append(cosine_similarity)
        length_difference_per_frame.append(length_difference)
        difference_per_frame.append(difference)
        mean_per_frame.append(neighbourhood_mean)
        #stdev_per_frame.append(std_dev)
    #continue
    neighbourhood_similarity.append(similarity_per_frame)
    neighbourhood_length_difference.append(length_difference_per_frame)
    neighbourhood_difference.append(difference_per_frame)
    neighbourhood_means.append(mean_per_frame)
    #neighbourhood_stdevs.append(stdev_per_frame)

In [ ]:
corrected_disps = []
corrected = []

for i, disps_per_frame in enumerate(disps):
    corrected_disps_per_frame = []
    corrected_per_frame = []

    for j, disp in enumerate(disps_per_frame):
        #if neighbourhood_similarity[i][j] < 0.95 and neighbourhood_length_difference[i][j] > 3.0:
        #    corrected_disps_per_frame.append(neighbourhood_means[i][j])
        #    corrected_per_frame.append(1.0)

        if neighbourhood_length_difference[i][j] > 4.0:
            corrected_disps_per_frame.append(neighbourhood_means[i][j])
            corrected_per_frame.append(1.0)
        elif neighbourhood_difference[i][j] > 4.0:
            corrected_disps_per_frame.append(neighbourhood_means[i][j])
            corrected_per_frame.append(1.0)
        elif neighbourhood_similarity[i][j] < 0.9:
            corrected_disps_per_frame.append(neighbourhood_means[i][j])
            corrected_per_frame.append(1.0)
        #elif 80 < i and corrected[i - 1][j] > 0.5:
        #    corrected_disps_per_frame.append(neighbourhood_means[i][j])
        #    corrected_per_frame.append(1.0)
        else:
            corrected_disps_per_frame.append(disps[i][j])
            corrected_per_frame.append(0.0)
    
    corrected_disps.append(corrected_disps_per_frame)
    corrected.append(corrected_per_frame)

In [ ]:
for i, corrected_disps_per_frame in enumerate(corrected_disps):
    for j, corrected_disp in enumerate(corrected_disps_per_frame):
        if neighbourhood_length_difference[i][j] > 3.0:
            corrected_disps[i][j] = neighbourhood_means[i][j]
            corrected[i][j] = 1.0
        elif neighbourhood_difference[i][j] > 3.0:
            corrected_disps[i][j] = neighbourhood_means[i][j]
            corrected[i][j] = 1.0
        elif neighbourhood_similarity[i][j] < 0.9:
            corrected_disps[i][j] = neighbourhood_means[i][j]
            corrected[i][j] = 1.0

In [ ]:
pl = pv.Plotter()
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=7, render_points_as_spheres=True, name='middle')

nd = disps[0].shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])
corrected_lines = np.hstack([[2, i, nd+i] for i in range(nd)])

vertices = np.vstack([centers, centers + disps[0]])
ll = pv.PolyData(vertices, lines=lines)
ll['corrected'] = corrected[0]
pl.add_mesh(ll, name='line', scalars='corrected', cmap='bwr')

corrected_vertices = np.vstack([centers, centers + corrected_disps[0]])
corrected_ll = pv.PolyData(corrected_vertices, lines=corrected_lines)
corrected_ll['corrected'] = corrected[0]
pl.add_mesh(corrected_ll, name='corrected_line', scalars='corrected', cmap='winter')

def callback(x):
    vertices = np.vstack([centers, centers + disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    ll['corrected'] = corrected[x]
    pl.add_mesh(ll, name='line', scalars='corrected', cmap='bwr')

    corrected_vertices = np.vstack([centers, centers + corrected_disps[x]])
    corrected_ll = pv.PolyData(corrected_vertices, lines=corrected_lines)
    corrected_ll['corrected'] = corrected[x]
    pl.add_mesh(corrected_ll, name='corrected_line', scalars='corrected', cmap='winter')
    
    pl.update()

pl.show()
interact(callback, x=(0, len(disps)-1, 1))



In [ ]:
fig, ax = mplt.subplots()
ax.set_xlim(0.0, 35.0)
*_, patches = ax.hist(np.linalg.norm(disps[0], axis=1), bins=50, color='blue')

def update_histogram(frame):
    global patches
    for p in patches:
        p.remove()

    *_, patches = ax.hist(np.linalg.norm(disps[frame], axis=1), bins=50, color='blue')
    return patches

ani = animation.FuncAnimation(fig=fig, func=update_histogram, frames=len(disps), interval=30)
ani.save("disp_length_over_time.gif")

In [ ]:
fig, ax = mplt.subplots()
ax.set_xlim(-30.0, 10.0)
ax.set_ylim(0, 200)

*_, x_patches = ax.hist(disps[0][:, 0], bins=50, alpha=0.33, color='blue')
*_, y_patches = ax.hist(disps[0][:, 1], bins=50, alpha=0.33, color='red')
*_, z_patches = ax.hist(disps[0][:, 2], bins=50, alpha=0.33, color='green')

def update_histogram(frame):
    global x_patches
    global y_patches
    global z_patches

    for p in x_patches:
        p.remove()
    for p in y_patches:
        p.remove()
    for p in z_patches:
        p.remove()

    *_, x_patches = ax.hist(disps[frame][:, 0], bins=50, alpha=0.33, color='blue')
    *_, y_patches = ax.hist(disps[frame][:, 1], bins=50, alpha=0.33, color='red')
    *_, z_patches = ax.hist(disps[frame][:, 2], bins=50, alpha=0.33, color='green')
    return (x_patches, y_patches, z_patches)

ani = animation.FuncAnimation(fig=fig, func=update_histogram, frames=len(disps), interval=30)
ani.save("disps_over_time.gif")
#mplt.show()


In [ ]:
def align_zoe(centers, disps):
    Q = centers
    P = (centers + disps)

    print(P[0:5])
    print(Q[0:5])

    #P = Q.copy()  # Example adjustment
    #for i in range(P.shape[0]):
    #    P[i, :] = P[i, :] + np.array([1, 2, 3])
    #P = 100 * P

    assert P.shape == Q.shape
    
    centeredP = P - P.mean(axis=0)
    centeredQ = Q - Q.mean(axis=0)

    scale_x = np.linalg.lstsq(centeredP[:, 0].reshape(-1, 1), centeredQ[:, 0])[0]
    scale_y = np.linalg.lstsq(centeredP[:, 1].reshape(-1, 1), centeredQ[:, 1])[0]
    scale_z = np.linalg.lstsq(centeredP[:, 2].reshape(-1, 1), centeredQ[:, 2])[0]

    S = np.array([[scale_x[0], 0.0, 0.0],
                  [0.0, scale_y[0], 0.0],
                  [0.0, 0.0, scale_z[0]]])

    t = Q.mean(axis=0) - P.mean(axis=0).dot(S)

    return S, t

align_zoe(centers, disps[100])

In [ ]:
def compute_align_disp(centers, disps, index):
    # Exclude outliers
    #magnitudes = np.linalg.norm(disps[index], axis=1)
    #z_scores = (magnitudes - np.mean(magnitudes)) / np.std(magnitudes)
    #include = abs(z_scores) < 3.0
    #print(include)

    # C++ code used igl procrustes for this... does seem like there is some rotation actually...
    M, t = align_zoe(centers, disps[index])

    tmp = centers + disps[index]
    tmp1 = (M @ tmp.T).T + t

    return tmp1-centers, M, t

    # Return the average disps
    #return (M @ centers.T).T + t - centers, M, t

compute_align_disp(centers, disps, 130)

In [ ]:
new_disps = []
Ms = []
ts = []
for i in range(len(disps)):
    d, M, t = compute_align_disp(centers, disps, i)
    new_disps.append(d)
    Ms.append(M)
    ts.append(t)

In [ ]:
new_corrected_disps = []
corrected_Ms = []
corrected_ts = []
for i in range(len(corrected_disps)):
    d, M, t = compute_align_disp(centers, corrected_disps, i)
    new_corrected_disps.append(d)
    Ms.append(M)
    ts.append(t)

In [ ]:
pv.set_jupyter_backend('trame')

pl = pv.Plotter()
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = disps[0].shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])
corrected_lines = np.hstack([[2, i, nd+i] for i in range(nd)])

corrected_vertices = np.vstack([centers, centers + new_corrected_disps[0]])
corrected_ll = pv.PolyData(corrected_vertices, lines=corrected_lines)
corrected_ll['corrected'] = corrected[0]
pl.add_mesh(corrected_ll, name='corrected_line', scalars='corrected', cmap='winter')
#pl.add_mesh(corrected_ll, name='corrected_line', color='red')

vertices = np.vstack([centers, centers + new_disps[0]])
ll = pv.PolyData(vertices, lines=lines)
ll['corrected'] = corrected[0]
pl.add_mesh(ll, name='line', scalars='corrected', cmap='bwr')
#pl.add_mesh(ll, name='linea', color='green')

def callback(x):
    corrected_vertices = np.vstack([centers, centers + new_corrected_disps[int(x)]])
    corrected_ll = pv.PolyData(corrected_vertices, lines=corrected_lines)
    corrected_ll['corrected'] = corrected[int(x)]
    pl.add_mesh(corrected_ll, name='corrected_line', scalars='corrected', cmap='winter')
    #pl.add_mesh(corrected_ll, name='corrected_line', color='red')

    vertices = np.vstack([centers, centers + new_disps[int(x)]])
    ll = pv.PolyData(vertices, lines=lines)
    ll['corrected'] = corrected[int(x)]
    pl.add_mesh(ll, name='line', scalars='corrected', cmap='bwr')
    #pl.add_mesh(ll, name='linea', color='green')
    #pl.update()

    #print(Ms[int(x)])
    #print(ts[int(x)])

pl.add_slider_widget(callback, (0, len(disps)-1))
pl.show()
#interact(callback, x=(0, len(disps)-1, 1), continuous_update=False)




In [ ]:
f=206
mplt.hist(np.linalg.norm(np.array(disps[f]-new_disps[f]), axis=1), bins=50)
mplt.show()

In [ ]:
data = [1/np.linalg.det(M) for M in Ms]
mplt.plot(data)
mplt.xlabel('time')
mplt.ylabel('scale')
mplt.show()

In [ ]:
data = [1/M[0,0] for M in Ms]
mplt.plot(data, label='x')

data = [1/M[1,1] for M in Ms]
mplt.plot(data, label='y')

data = [1/M[2,2] for M in Ms]
mplt.plot(data, label='z')

mplt.xlabel('time')
mplt.ylabel('scale')
mplt.legend()
mplt.title('Scale factors')
mplt.show()

test = [1/M[2,2] for M in Ms]
print(test[72])

In [ ]:
np.mean(np.linalg.norm(disps[-1][:,:2], axis=1)), np.mean(np.linalg.norm(new_disps[-1][:,:2], axis=1))

In [ ]:
disps[-1][:,:2]

In [ ]:
print(ts)
data = [t[0] for t in ts]
mplt.plot(data, label='x')
data = [t[1] for t in ts]
mplt.plot(data, label='y')
data = [t[2] for t in ts]
mplt.plot(data, label='z')
mplt.legend()
mplt.xlabel('time')
mplt.ylabel('translation')
mplt.show()

In [ ]:
data = [np.mean(t[:, 0]) for t in new_disps]
mplt.plot(data, label='x')
data = [np.mean(t[:, 1]) for t in new_disps]
mplt.plot(data, label='y')
data = [np.mean(t[:, 2]) for t in new_disps]
mplt.plot(data, label='z')
mplt.legend()
mplt.xlabel('time')
mplt.ylabel('disp')
mplt.title('Mean disp after correction')
mplt.show()

In [ ]:
data = [np.mean(t[:, 0]) for t in disps]
mplt.plot(data, label='x')
data = [np.mean(t[:, 1]) for t in disps]
mplt.plot(data, label='y')
data = [np.mean(t[:, 2]) for t in disps]
mplt.plot(data, label='z')
mplt.legend()
mplt.xlabel('time')
mplt.ylabel('disp')
mplt.title('Mean disp before correction')
mplt.show()

In [ ]:
from scipy.interpolate import RBFInterpolator


In [ ]:
default_json = { 
"geometry": {
        "mesh": "___",
        "volume_selection": 1
    },
"boundary_conditions": {
    "dirichlet_boundary": "___"
},
"materials": {
        "E": E,
        "id": 1,
        "nu": nu,
        "type": "LinearElasticity" if is_linear else "NeoHookean"
},
"output": {
    "json": "___",
    "directory": "___",
    "paraview": {
        "file_name": "___",
        "surface": True,
        "options": {
            "material": True,
            "forces": True
        },
        "vismesh_rel_area": 10000000
    }
}
}

In [ ]:
def generate_json(out, V, T, middle, bottom, top, centers, disps, index, eps):
    rbf = RBFInterpolator(centers, disps[index])
    disp = rbf(V[middle,:])

    with open(f"{out}disp_{index}.txt", "w") as f:
        for i in range(middle.shape[0]):
            f.write(f"{middle[i]} {disp[i, 0]} {disp[i, 1]} {disp[i, 2]}\n")
        for i in range(bottom.shape[0]):
            f.write(f"{bottom[i]} 0 0 0\n")
        for i in range(top.shape[0]):
            f.write(f"{top[i]} 0 0 0\n")

    mesh = mio.Mesh(points=V, cells={"tetra": T})
    mesh.write(f"{out}mesh.msh", file_format="gmsh")

    json = default_json.copy()
    json["geometry"]["mesh"] = f"mesh.msh"
    json["boundary_conditions"]["dirichlet_boundary"] = f"disp_{index}.txt"
    json["output"]["directory"] = out
    json["output"]["json"] = f"sim{index}.json"
    json["output"]["paraview"]["file_name"] = f"sim{index}.vtu"

    with open(f"{out}run_{index}.json", "w") as f:
        js.dump(json, f, indent=4)



generate_json("outnz/", V,T, middle, bottom, top, centers, disps, 239, eps)

In [ ]:
for i in range(len(disps)):
    generate_json("outnz/", V,T, middle, bottom, top, centers, new_disps, i, eps)

In [ ]:
index=200
sup.run([polyfem, "-j", f"outnz/run_{index}.json"], check=True)

In [ ]:
index = 212
corrected_tf = [val < 0.5 for val in corrected[index]]
rbf = RBFInterpolator(centers[corrected_tf], new_disps[index][corrected_tf])
rbf_disp = rbf(centers[corrected_tf])
    # ^^ replace with right disps :) and add to visualization
print(rbf_disp.shape)
print(centers[corrected_tf].shape)



In [ ]:
pv.set_jupyter_backend('trame')

pl = pv.Plotter()
pl.add_mesh(to_pyvista_mesh(centers[corrected_tf]), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = rbf_disp.shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])

vertices = np.vstack([centers[corrected_tf], centers[corrected_tf] + rbf_disp])
ll = pv.PolyData(vertices, lines=lines)
#ll['corrected'] = corrected[0]
pl.add_mesh(ll, name='line', color='green')

#def callback(x):
#    vertices = np.vstack([centers, centers + new_disps[int(x)]])
#    ll = pv.PolyData(vertices, lines=lines)
#    ll['corrected'] = corrected[int(x)]
#    pl.add_mesh(ll, name='line', color='green')


#pl.add_slider_widget(callback, (0, len(disps)-1))
pl.show()